Ce notebook permet de transformer notre fichier .csv contenant les résultats des prédictions en un fichier .json qui est le format attendu pour la phase d'évaluation.

In [ ]:
import json
import os

# Chemin d'entrée
input_path = r"C:/Users/Guidi/Desktop/Ecole_Chartes/Python/map_projet/mapreader_project/IGN_data/ign25synth.json"

# Chemin de sortie
output_path = r"C:/Users/Guidi/Desktop/Ecole_Chartes/Python/map_projet/mapreader_project/IGN_data/metadata_500_images.json"

# Charger le fichier JSON complet
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Garder uniquement les 20004 premières images
subset = data[:500]

# Sauvegarder dans le chemin de sortie
with open(output_path, "w", encoding="utf-8") as f_out:
    json.dump(subset, f_out, ensure_ascii=False, indent=2)

print(f"Fichier créé youpiii : {output_path}")


Fichier créé youpiii : C:/Users/Guidi/Desktop/Ecole_Chartes/Python/map_projet/mapreader_project/IGN_data/metadata_500_images.json


Première étape : Transformer notre fichier .csv en fichier .json. Le but est d'obtenir en contenu une liste de dictionnaires pour chaque image contenant pour chacune les mots annotés avec leurs coordonnées polygonales et leur texte.

In [ ]:
import pandas as pd
import json
from shapely.wkt import loads

# On charge le csv
df = pd.read_csv(r"C:\Users\Guidi\Desktop\Ecole_Chartes\Python\evaluation\parent_predictions_sample.csv")

#Petite fonction pour transformer les polygones en liste de coordonnées (ce qui est attendu pour l'évaluation)
#Pour se faire, on extrait les sommets des polygones donnés au format WKT en supprimant le doublon de fermeture (un sommet apparait en effet en doublons puisque le polygone a une forme fermée).
#On retourne une liste toute propre de coordonnées [x, y] représentant son contour.
def polygon_to_vertices(polygon_wkt):
    polygon = loads(polygon_wkt)
    coords = list(polygon.exterior.coords)
    if coords[0] == coords[-1]:
        coords = coords[:-1]
    return [[x, y] for x, y in coords]

result = []
for image_id, group in df.groupby("image_id"): # On groupe par image_id toutes els annotations associées
    words = []
    for _, row in group.iterrows():# On itère sur chaque ligne du groupe
        vertices = polygon_to_vertices(row["pixel_geometry"]) #petite application de la fonction vue au-dessus
        word = {"vertices": vertices, "text": str(row["text"])}# On crée un dictionnaire pour chaque mot avec ses coordonnées et son texte
        words.append(word)
    result.append({"image": image_id, "groups": [words]})

# On écrit le résultat dans un fihcier json
with open(r"C:\Users\Guidi\Desktop\Ecole_Chartes\Python\evaluation\resultatsample.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)#nous avons demandé à chatgpt pour les paramètres à choisir


Pour l'évaluation, le fichier. json de vérité de terrain va être comparé au fichier .json des prédictions. Nous veillons à ce que les clés, les noms d'images soient identiques pour la comparaison.

Pour le fichier .json que nous avons créé, l'image ne comprend pas de chemin relatif dans son nom mais le fichier .json de vérité de terrain en contient. On le supprime.

In [4]:

# On définit chemin entrée et de sortie
input_file = r'C:\Users\Guidi\Desktop\Ecole_Chartes\Python\evaluation\metadata_sample.json'
output_file = r'C:\Users\Guidi\Desktop\Ecole_Chartes\Python\evaluation\metadatagoodsample.json'

# On charge le JSON
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

# On fait en sorte de ne garder que le nom des images sans préfixe
for entry in data:
    full_path = entry.get('image', '') #on extrait le nom de l'image et on rajoute la sécurité .get pour éviter une erreur si la clé n'existe pas
    file_name = full_path.split('/')[-1]  #  On garde juste le nom fichier en enlevant le chemin  qui le préfixe
    entry['image'] = file_name  # # On remplace le champ image avec juste le nom du fichier

# Sauvegarde du fichier
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)


C'est bon, les deux fichiers sont prêts à être comparés !!